# DPO (Direct Preference Optimization)

**Tags:** finetuning, alignment
**Prerequisites:** RLHF (Reinforcement Learning from Human Feedback)
**Related Concepts:** See flowchart below
**Source:** llm/concepts/dpo.md

## TL;DR

Fine-tune LLM directly using human preference pairs, without training separate reward model. Eliminates RLHF's reward model + RL training pipeline. Simpler, faster, fewer hyperparameters, comparable quality to RLHF. Single SFT stage instead of SFT → reward model → PPO.

## Core Intuition

RLHF trains two models (reward model + policy) using RL, which is complex and unstable. DPO realizes you can extract preference signals directly via loss function: if humans prefer output A over B for prompt X, optimize LLM to assign higher likelihood to A. No separate reward model needed. Single stage fine-tuning.

## How It Works

**RLHF Pipeline (Traditional):**
```
1. SFT: Fine-tune LLM on (prompt, response) pairs
2. Reward Model Training:
   - Collect (prompt, A, B) where A > B (human preference)
   - Train reward model: r(x, y) → scalar score
   - Optimize: minimize -log(σ(r(x, y_A) - r(x, y_B)))
3. RL Fine-tuning (PPO):
   - Use reward model as signal
   - RL objective: maximize E[reward] - KL(policy || base_policy)
   - Multiple gradient steps per sample
```

**DPO Pipeline (Direct):**
```
1. SFT: Fine-tune LLM on (prompt, response) pairs [optional]
2. Collect preference data: (prompt, y_preferred, y_dispreferred)
3. DPO Loss (single stage):
   Optimize: minimize -log(σ(β * (log p(y_w | x) - log p(y_l | x))))
   where σ is logistic, β is temperature
```

**DPO Loss Derivation:**
```
Goal: maximize human preference likelihood
P(y_w > y_l | x) ∝ exp(r(x, y_w) - r(x, y_l))

Implicit reward model:
r(x, y) = (β / (1 - π_SFT)) * log(π_θ(y | x) / π_ref(y | x))

DPO Loss:
L_DPO = -log(σ(β * log(π(y_w|x) / π_ref(y_w|x)) - β * log(π(y_l|x) / π_ref(y_l|x))))

Where:
- π(y|x): policy (model being optimized)
- π_ref(y|x): reference (SFT model)
- β: temperature controlling preference strength
- σ: logistic function
```

**Key Insight:**
DPO extracts reward model implicitly. No need to:
- Train separate reward model (saves compute)
- Run PPO (unstable RL training)
- Tune RL hyperparameters (learning rate, KL penalty, etc.)

**Algorithm:**
```
Input: Preference pairs {(x, y_w, y_l)}, base SFT model π_ref, β (temperature)

For each preference pair:
  1. Forward pass on preferred response: log_probs_w = π(y_w | x)
  2. Forward pass on dispreferred response: log_probs_l = π(y_l | x)
  3. Reference forward passes: log_probs_ref_w, log_probs_ref_l
  4. Compute DPO loss
  5. Backward pass, update π_θ
```

### Workflow Diagram**Note:** This is a basic workflow template. Review and customize based on specific concept.
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

## Key Properties & Trade-offs

/ Trade-offs

| Method | Stages | Reward Model | Complexity | Speed | Quality | Stability |
|--------|--------|--------------|-----------|-------|---------|-----------|
| SFT | 1 | None | Low | Fast | 70-80% | High |
| RLHF | 3 (SFT+RM+PPO) | Yes (separate) | High | Slow (weeks) | 90-95% | Medium |
| DPO | 2 (SFT+DPO) | Implicit | Medium | Fast (days) | 90-95% | High |
| IPO | 2 (SFT+IPO) | Implicit | Medium | Fast | 91-96% | Very High |

**DPO Advantages:**
- Simpler: single supervised loss, no RL
- Faster: days vs weeks
- Fewer hyperparameters: just β (vs KL penalty, learning rate, etc.)
- Stable: gradient-based, not policy optimization
- Computationally cheaper: one forward pass per pair

**DPO Disadvantages:**
- Implicit reward model (harder to inspect/debug)
- Requires good SFT base (quality matters)
- β hyperparameter critical (affects preference strength)
- Less tested at scale than RLHF

### Code Implementation

```python
# Key imports for DPO (Direct Preference Optimization)
import numpy as np
import torch
from typing import Any

# DPO (Direct Preference Optimization) example implementation
class DPO(DirectPreferenceOptimization):
    """
    DPO (Direct Preference Optimization) implementation.
    This is a template - customize with actual code.
    """
    def __init__(self):
        pass

    def process(self, input_data: Any) -> Any:
        # Interview tip: Explain the core insight here
        return input_data
```

### Mathematical Formulation

Include LaTeX equations relevant to this concept.

**Example:**
$$\text{Output} = f(\text{Input})$$

## Related Concepts
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

### Common Interview Questions

**Q: What is DPO (Direct Preference Optimization) used for?**
A: [Add concise answer about practical application]

**Q: What are the main trade-offs of DPO (Direct Preference Optimization)?**
A: [Discuss pros/cons and when to use vs alternatives]

**Q: How does DPO (Direct Preference Optimization) compare to [related concept]?**
A: [Explain key differences and when to use each]

**Q: What are common mistakes when using DPO (Direct Preference Optimization)?**
A: [List 1-2 common pitfalls and how to avoid them]

**Q: Can you explain the intuition behind DPO (Direct Preference Optimization)?**
A: [Provide a simple analogy or explanation]

## References

- **Source Document:** `llm/concepts/dpo.md`
- **Related Papers:** [Add relevant papers]
- **Implementations:**
  - HuggingFace: [Add links]
  - GitHub: [Add links]